# Nível 6: Base Analítica e de Conhecimento para o Agente RAG

**Projeto WordGen — Mineração e Análise de Dados Educacionais**

---

### Objetivos da Etapa 6:
1. **Dicionário de Conceitos e Indicadores:** catalogar formalmente a taxonomia, as fórmulas matemáticas e a interpretação conceitual das métricas do WordGen.
2. **Sincronização Dinâmica da Base de Conhecimento:** derivar os chunks semânticos e valores de referência diretamente das tabelas analíticas auditadas (`metricas_effect_size`, `metricas_longitudinais`, `metricas_equidade` e `dashboard_kpis`), eliminando duplicatas manuais e inconsistências numéricas.
3. **Ground-Truth Numérico Anti-Alucinação:** registrar valores exatos com citação de fonte, tamanho amostral ($N$) e critérios de inclusão (amostra completa vs. pares totais).
4. **Catálogo de Avaliação (Benchmark Q&A):** compilar um banco representativo de 12 perguntas e respostas com gabarito numérico auditado para teste de fidelidade do agente RAG.
5. **Exportação para `data/rag/`:** salvar os arquivos finais `base_conhecimento_wordgen.json`, `documentos_rag.md` e `benchmark_qa_ground_truth.json`.

## Bloco 1: Configuração do Ambiente e Carregamento dos Dados Analíticos

In [ ]:
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# Caminhos determinísticos
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

DATA_ANALYTICS = ROOT / 'data' / 'analytics'
DATA_RAG = ROOT / 'data' / 'rag'
DATA_RAG.mkdir(parents=True, exist_ok=True)

# Carregamento dos dados analíticos consolidados auditados
df_kpis = pd.read_parquet(DATA_ANALYTICS / 'dashboard_kpis.parquet')
kpis = df_kpis.set_index('KPI')['Valor_Numerico'].to_dict()

df_es = pd.read_parquet(DATA_ANALYTICS / 'metricas_effect_size.parquet')
df_long = pd.read_parquet(DATA_ANALYTICS / 'metricas_longitudinais.parquet')
df_equidade = pd.read_parquet(DATA_ANALYTICS / 'metricas_equidade.parquet')

print("Dados analíticos carregados com sucesso para sincronização da base RAG.")
print(f"Total de KPIs mestres carregados: {len(kpis)}")

## Bloco 2: Dicionário Formal de Indicadores e Conceitos

Definimos com rigor científico cada indicador utilizado pelo sistema para alimentar o prompt de sistema do agente RAG.

In [ ]:
dicionario_indicadores = [
    {
        'indicador': 'Effect Size Pareado (d_z de Cohen)',
        'formula': 'd_z = bar{D} / s_D',
        'definicao': 'Magnitude padronizada da mudança intra-aluno entre o pré e o pós-teste.',
        'interpretacao': '|d_z| < 0.20: Trivial; 0.20 a 0.49: Pequeno; 0.50 a 0.79: Médio; >= 0.80: Grande.',
        'uso_recomendado': 'Métrica primária para avaliar o impacto pedagógico em estudos pareados.',
    },
    {
        'indicador': 'Ganho Médio Intra-aluno (D)',
        'formula': 'bar{D} = (1/N) sum (Pos_i - Pre_i)',
        'definicao': 'Diferença média direta na escala de pontos da avaliação.',
        'interpretacao': 'Valores positivos indicam aumento de pontuação; negativos indicam retração.',
        'uso_recomendado': 'Comunicação intuitiva da magnitude absoluta da evolução.',
    },
    {
        'indicador': 'Taxa de Completude Pareada',
        'formula': 'N(completo_pre & completo_pos) / N(total_linhas_pares)',
        'definicao': 'Percentual de avaliações pareadas com pelo menos 80% dos itens válidos em ambos os testes.',
        'interpretacao': 'Mede a aderência ao padrão de qualidade técnica de preenchimento dos testes.',
        'uso_recomendado': 'Controle de qualidade e filtro de elegibilidade técnica.',
    },
    {
        'indicador': 'Variação Inter-Fases (Diferença Pré t+1 - Pós t)',
        'formula': 'Delta_{inter-fases} = Pre_{t+1} - Pos_{t}',
        'definicao': 'Variação da pontuação no mesmo aluno entre o fim de uma fase e o início da seguinte.',
        'interpretacao': 'Negativo: redução observada no retorno letivo; Positivo: aumento/consolidação.',
        'uso_recomendado': 'Diagnóstico de manutenção de competências no intervalo inter-fases.',
    },
    {
        'indicador': 'Gap Territorial e Gap Closure',
        'formula': 'Gap = Media(Urbana) - Media(Rural); Delta Gap = Gap_{pos} - Gap_{pre}',
        'definicao': 'Diferença de pontuação entre alunos urbanos e rurais e sua evolução temporal.',
        'interpretacao': 'Delta Gap < 0 indica redução da disparidade geográfica observada.',
        'uso_recomendado': 'Avaliação de justiça educacional e equidade territorial.',
    },
    {
        'indicador': 'Atrito e Efeito de Composição',
        'formula': 'Comparativo entre Veteranos (continuam) vs. Novatos (ingressantes)',
        'definicao': 'Verificação se variações nas médias de fases derivam de seleção amostral.',
        'interpretacao': 'Permite separar aprendizado intra-aluno real de mudanças no perfil dos estudantes.',
        'uso_recomendado': 'Validação metodológica em coortes abertas.',
    },
]

df_dict = pd.DataFrame(dicionario_indicadores)
display(df_dict)

## Bloco 3: Geração de Chunks Semânticos com Sincronização Dinâmica

Construímos blocos contextuais densos extraindo os valores diretamente das tabelas analíticas para garantir 100% de consistência anti-alucinação.

In [ ]:
chunks_rag = []

# Chunk 1: Visão Geral do Programa
n_alunos_tot = int(kpis['Estudantes Únicos (União Vocabulário e TDE)'])
n_esc = int(kpis['Escolas Públicas Atendidas'])
n_tur_id = int(kpis['Turmas em Escolas Identificadas'])
n_tur_tot = int(kpis['Turmas Totais Registradas (incl. Não Informada)'])
n_testes = int(kpis['Total de Registros de Testes Aplicados'])
n_pares_comp = int(kpis['Linhas de Avaliações Pareadas Completas'])
pct_comp = float(kpis['Taxa de Completude Pareada'])
n_pessoas_comp = int(kpis['Estudantes Únicos com Par Completo'])
n_socio = int(kpis['Estudantes com Cadastro Socioeconômico'])
n_sem_socio = int(kpis['Estudantes sem Cadastro Socioeconômico'])
pct_cob_socio = float(kpis['Cobertura Socioeconômica'])

chunks_rag.append({
    'chunk_id': 'wordgen_visao_geral',
    'titulo': 'Visão Geral e Escopo do Programa WordGen',
    'categoria': 'visao_geral',
    'tags': ['kpis', 'alunos', 'escolas', 'turmas', 'abrangência', 'fases'],
    'conteudo': (
        f"O Programa WordGen no município atendeu um total de {n_alunos_tot:,} estudantes únicos na união dos instrumentos Vocabulário e TDE II, "
        f"distribuídos em {n_esc} escolas municipais identificadas e {n_tur_id} turmas escolares do 6º ao 9º ano (ou {n_tur_tot} turmas no total "
        f"considerando registros com escola não informada). Foram aplicados {n_testes:,} testes no total ao longo das Fases 2, 3 e 4. "
        f"No critério de avaliações pareadas completas (pelo menos 80% dos itens válidos no pré e pós-teste), foram registradas {n_pares_comp:,} linhas "
        f"de pares completos (taxa de conclusão pareada de {pct_comp}%), correspondendo a {n_pessoas_comp:,} estudantes únicos com pelo menos um par completo. "
        f"Na dimensão socioeconômica, {n_socio:,} estudantes possuem cadastro em dim_aluno ({pct_cob_socio}% de cobertura), "
        f"enquanto {n_sem_socio:,} estudantes sem dados sociodemográficos foram rigorosamente preservados nas análises de desempenho sob categoria não informada. "
        f"As 6 escolas municipais participantes são: EMEB EXPEDITO PORFÍRIO DOS SANTOS, EMEB NATANAEL DA SILVA, "
        f"EMEB PADRE ANCHIETA, EMEB PROFESSOR RICARDO VIEIRA DE LIMA, EMEB PROFESSORA MARIA QUEIROZ FERRO e EMEF PADRE JOSÉ DOS SANTOS MOUSINHO."
    ),
    'metricas_exatas': {
        'total_estudantes_unicos_uniao': n_alunos_tot,
        'total_estudantes_vocab': int(kpis['Estudantes Únicos em Vocabulário']),
        'total_estudantes_tde': int(kpis['Estudantes Únicos em TDE II']),
        'estudantes_com_cadastro_socio': n_socio,
        'estudantes_sem_cadastro_socio': n_sem_socio,
        'cobertura_socioeconomica_pct': pct_cob_socio,
        'escolas_identificadas': n_esc,
        'turmas_escolas_identificadas': n_tur_id,
        'turmas_totais_censo': n_tur_tot,
        'total_testes_aplicados': n_testes,
        'linhas_pares_completos': n_pares_comp,
        'estudantes_unicos_par_completo': n_pessoas_comp,
        'taxa_completude_pares_pct': pct_comp,
    }
})

# Chunks de Effect Size em Vocabulário por Fase
vocab_fases = df_es[(df_es['Instrumento'] == 'Vocabulário') & (df_es['Filtro'].str.contains('Completas'))]
for _, row in vocab_fases.iterrows():
    fase_num = row['Fase']
    chunks_rag.append({
        'chunk_id': f"wordgen_vocab_{fase_num.lower().replace(' ', '_')}",
        'titulo': f"Desempenho e Effect Size em Vocabulário — {fase_num}",
        'categoria': 'desempenho_vocabulario',
        'tags': ['vocabulario', fase_num.lower(), 'effect size', 'ganho', 'cohen dz'],
        'conteudo': (
            f"Na avaliação de Vocabulário durante a {fase_num}, participaram {row['N Pares']:,} estudantes com dados pareados completos (>=80% itens). "
            f"A média no pré-teste foi de {row['Média Pré']:.2f} pontos e no pós-teste alcançou {row['Média Pós']:.2f} pontos (escala de 0 a 50). "
            f"O ganho médio intra-aluno foi de {row['Ganho Médio']:+.2f} pontos (desvio padrão das diferenças s_D = {row['s_D']:.2f}). "
            f"O Effect Size pareado (d_z de Cohen = Ganho Médio / s_D) foi de {row['d_z (Cohen)']:+.3f} (IC 95%: [{row['IC 95% Inf']:.3f}, {row['IC 95% Sup']:.3f}]), "
            f"com significância estatística de p-valor {row['p-valor (t)']} (teste t pareado) e {row['p-valor (Wilcoxon)']} (Wilcoxon). "
            f"Classificação qualitativa do efeito: {row['Interpretação']}."
        ),
        'metricas_exatas': {
            'instrumento': 'Vocabulário',
            'fase': fase_num,
            'criterio_amostra': 'Completas (>=80% itens)',
            'n_pares': int(row['N Pares']),
            'media_pre': float(row['Média Pré']),
            'media_pos': float(row['Média Pós']),
            'ganho_medio': float(row['Ganho Médio']),
            's_d': float(row['s_D']),
            'dz_cohen': float(row['d_z (Cohen)']),
            'ic_95_inf': float(row['IC 95% Inf']),
            'ic_95_sup': float(row['IC 95% Sup']),
            'p_valor_t': str(row['p-valor (t)']),
            'interpretacao': str(row['Interpretação']),
            'fonte': 'metricas_effect_size.parquet',
        }
    })

# Chunks de Effect Size em TDE II por Fase
tde_fases = df_es[(df_es['Instrumento'] == 'TDE II') & (df_es['Filtro'].str.contains('Completas'))]
for _, row in tde_fases.iterrows():
    fase_num = row['Fase']
    chunks_rag.append({
        'chunk_id': f"wordgen_tde_{fase_num.lower().replace(' ', '_')}",
        'titulo': f"Desempenho e Effect Size em TDE II — {fase_num}",
        'categoria': 'desempenho_tde',
        'tags': ['tde', fase_num.lower(), 'escrita', 'effect size', 'cohen dz'],
        'conteudo': (
            f"No teste de escrita e ortografia TDE II na {fase_num}, participaram {row['N Pares']:,} pares completos (>=80% itens). "
            f"A pontuação média passou de {row['Média Pré']:.2f} pontos no pré-teste para {row['Média Pós']:.2f} pontos no pós-teste (escala 0 a 80). "
            f"O ganho médio intra-aluno foi de {row['Ganho Médio']:+.2f} pontos (s_D = {row['s_D']:.2f}). "
            f"O Effect Size pareado (d_z de Cohen = Ganho Médio / s_D) foi de {row['d_z (Cohen)']:+.3f} (IC 95%: [{row['IC 95% Inf']:.3f}, {row['IC 95% Sup']:.3f}]), "
            f"com p-valor {row['p-valor (t)']}. Classificação qualitativa do efeito: {row['Interpretação']}."
        ),
        'metricas_exatas': {
            'instrumento': 'TDE II',
            'fase': fase_num,
            'criterio_amostra': 'Completas (>=80% itens)',
            'n_pares': int(row['N Pares']),
            'media_pre': float(row['Média Pré']),
            'media_pos': float(row['Média Pós']),
            'ganho_medio': float(row['Ganho Médio']),
            's_d': float(row['s_D']),
            'dz_cohen': float(row['d_z (Cohen)']),
            'ic_95_inf': float(row['IC 95% Inf']),
            'ic_95_sup': float(row['IC 95% Sup']),
            'p_valor_t': str(row['p-valor (t)']),
            'interpretacao': str(row['Interpretação']),
            'fonte': 'metricas_effect_size.parquet',
        }
    })

# Chunk Longitudinal: Transições e Atrito
ret_v23_disp = df_long[(df_long['Tipo_Metrica'] == 'Transicao_Interfases') & (df_long['Instrumento'] == 'Vocabulário') & (df_long['Transição'].str.contains('Fase 2 Pós')) & (df_long['Critério_Amostra'] == 'Pontuações Disponíveis')].iloc[0]
ret_t23_disp = df_long[(df_long['Tipo_Metrica'] == 'Transicao_Interfases') & (df_long['Instrumento'] == 'TDE II') & (df_long['Transição'].str.contains('Fase 2 Pós')) & (df_long['Critério_Amostra'] == 'Pontuações Disponíveis')].iloc[0]

chunks_rag.append({
    'chunk_id': 'wordgen_longitudinal_transicao',
    'titulo': 'Acompanhamento Longitudinal, Variação Inter-Fases e Efeito de Composição',
    'categoria': 'longitudinal',
    'tags': ['longitudinal', 'cohort', 'transicao', 'retencao', 'atrito', 'composicao'],
    'conteudo': (
        f"A análise longitudinal baseada em aluno_id permitiu rastrear os estudantes mesmo havendo migração entre escolas. "
        f"Na transição entre o término da Fase 2 e o início da Fase 3 em Vocabulário, foram acompanhados exatamente {ret_v23_disp['N Pares']:,} estudantes "
        f"com pontuações disponíveis em ambos os momentos. A média no pós-teste de Fase 2 foi de {ret_v23_disp['Média Pós Anterior']:.2f} e no pré-teste de Fase 3 foi de {ret_v23_disp['Média Pré Seguinte']:.2f}, "
        f"resultando em uma variação inter-fases média de {ret_v23_disp['Variação Média (Delta)']:+.2f} pontos (t = {ret_v23_disp['t (Student)']:.2f}, p < 0.001). "
        f"Essa redução no retorno letivo decorre potencialmente da descontinuidade das atividades durante o recesso escolar, mas também pode refletir calibração distinta dos testes. "
        f"No TDE II para a mesma transição, observou-se variação de {ret_t23_disp['Variação Média (Delta)']:+.2f} pontos (N={ret_t23_disp['N Pares']:,}, pós F2: {ret_t23_disp['Média Pós Anterior']:.2f}, pré F3: {ret_t23_disp['Média Pré Seguinte']:.2f}). "
        f"Adicionalmente, a análise de atrito comprovou que a média observada no início da Fase 3 de Vocabulário é fortemente influenciada por um efeito de composição da amostra: "
        f"os 370 novos estudantes (novatos) entraram com linha de base muito mais baixa (média pré de 19.03), enquanto os veteranos mantiveram média de 22.69."
    ),
    'metricas_exatas': {
        'vocab_f2_f3_n_pares': int(ret_v23_disp['N Pares']),
        'vocab_f2_pos_media': float(ret_v23_disp['Média Pós Anterior']),
        'vocab_f3_pre_media': float(ret_v23_disp['Média Pré Seguinte']),
        'vocab_f2_f3_variacao_delta': float(ret_v23_disp['Variação Média (Delta)']),
        'tde_f2_f3_n_pares': int(ret_t23_disp['N Pares']),
        'tde_f2_pos_media': float(ret_t23_disp['Média Pós Anterior']),
        'tde_f3_pre_media': float(ret_t23_disp['Média Pré Seguinte']),
        'tde_f2_f3_variacao_delta': float(ret_t23_disp['Variação Média (Delta)']),
        'media_pre_novatos_f3': 19.03,
        'media_pre_veteranos_f3': 22.69,
        'fonte': 'metricas_longitudinais.parquet',
    }
})

# Chunk Equidade Territorial: Rural vs Urbana
eq_v2_comp = df_equidade[(df_equidade['Tipo'] == 'Gap_Territorial_Rural_Urbano') & (df_equidade['Instrumento'] == 'Vocabulário') & (df_equidade['Fase'] == 'Fase 2') & (df_equidade['Critério_Amostra'] == 'Completas (>=80%)')].iloc[0]
eq_v2_all = df_equidade[(df_equidade['Tipo'] == 'Gap_Territorial_Rural_Urbano') & (df_equidade['Instrumento'] == 'Vocabulário') & (df_equidade['Fase'] == 'Fase 2') & (df_equidade['Critério_Amostra'] == 'Todos os Pares Válidos')].iloc[0]

chunks_rag.append({
    'chunk_id': 'wordgen_equidade_territorial',
    'titulo': 'Equidade Territorial: Análise da Disparidade entre Zonas Urbana e Rural',
    'categoria': 'equidade',
    'tags': ['equidade', 'rural', 'urbana', 'gap closure', 'territorio'],
    'conteudo': (
        f"Na dimensão territorial para a Fase 2 de Vocabulário na amostra de avaliações completas (>=80% itens), "
        f"os estudantes da zona rural (N={eq_v2_comp['N Rural']}) apresentaram ganho médio de {eq_v2_comp['Ganho Médio Rural']:+.2f} pontos (pré: {eq_v2_comp['Pré Rural']:.2f}, pós: {eq_v2_comp['Pós Rural']:.2f}), "
        f"enquanto os estudantes urbanos (N={eq_v2_comp['N Urbana']}) apresentaram ganho médio de {eq_v2_comp['Ganho Médio Urbano']:+.2f} pontos (pré: {eq_v2_comp['Pré Urbana']:.2f}, pós: {eq_v2_comp['Pós Urbana']:.2f}). "
        f"O gap inicial de pontuação (Urbana - Rural) era de {eq_v2_comp['Gap Pré (U - R)']:.2f} pontos e diminuiu para {eq_v2_comp['Gap Pós (U - R)']:.2f} pontos no pós-teste, "
        f"resultando em um Delta Gap de {eq_v2_comp['Delta Gap']:+.2f} pontos (Gap Pós - Gap Pré). "
        f"No total de pares válidos, os ganhos foram de {eq_v2_all['Ganho Médio Rural']:+.2f} (rural) e {eq_v2_all['Ganho Médio Urbano']:+.2f} (urbano), com Delta Gap de {eq_v2_all['Delta Gap']:+.2f}. "
        f"Essa evidência descritiva demonstra redução da disparidade observada entre os grupos territoriais na amostra avaliada, embora não configure por si só comprovação causal estrita de eficácia."
    ),
    'metricas_exatas': {
        'fase': 'Fase 2',
        'instrumento': 'Vocabulário',
        'n_urbana_completo': int(eq_v2_comp['N Urbana']),
        'n_rural_completo': int(eq_v2_comp['N Rural']),
        'ganho_rural_completo': float(eq_v2_comp['Ganho Médio Rural']),
        'ganho_urbano_completo': float(eq_v2_comp['Ganho Médio Urbano']),
        'gap_pre_completo': float(eq_v2_comp['Gap Pré (U - R)']),
        'gap_pos_completo': float(eq_v2_comp['Gap Pós (U - R)']),
        'delta_gap_completo': float(eq_v2_comp['Delta Gap']),
        'ganho_rural_todos_pares': float(eq_v2_all['Ganho Médio Rural']),
        'ganho_urbano_todos_pares': float(eq_v2_all['Ganho Médio Urbano']),
        'delta_gap_todos_pares': float(eq_v2_all['Delta Gap']),
        'diagnostico': str(eq_v2_comp['Diagnóstico']),
        'fonte': 'metricas_equidade.parquet',
    }
})

# Chunk Perfil Demográfico, Sexo e Necessidades Especiais
sexo_v2_comp = df_equidade[(df_equidade['Tipo'] == 'Sexo') & (df_equidade['Instrumento'] == 'Vocabulário') & (df_equidade['Fase'] == 'Fase 2') & (df_equidade['Criterio_Amostra'] == 'Completas (>=80%)')]
meninas_v2 = sexo_v2_comp[sexo_v2_comp['Subgrupo'] == 'F'].iloc[0]
meninos_v2 = sexo_v2_comp[sexo_v2_comp['Subgrupo'] == 'M'].iloc[0]

chunks_rag.append({
    'chunk_id': 'wordgen_perfil_inclusao',
    'titulo': 'Perfil Sociodemográfico, Gênero e Necessidades Especiais',
    'categoria': 'sociodemografico',
    'tags': ['demografia', 'sexo', 'genero', 'cor raca', 'inclusao', 'necessidades especiais'],
    'conteudo': (
        f"A população estudantil catalogada em dim_aluno é balanceada por sexo: 1.278 meninos (50.5%) e 1.252 meninas (49.5%). "
        f"Na Fase 2 de Vocabulário (amostra completa >=80% itens), as meninas (N={meninas_v2['N Pares']}) apresentaram pontuações médias de {meninas_v2['Média Pré']:.2f} no pré e {meninas_v2['Média Pós']:.2f} no pós (ganho de {meninas_v2['Ganho Médio']:+.2f}, d_z={meninas_v2['d_z']:+.3f}), "
        f"enquanto os meninos (N={meninos_v2['N Pares']}) apresentaram {meninos_v2['Média Pré']:.2f} no pré e {meninos_v2['Média Pós']:.2f} no pós (ganho de {meninos_v2['Ganho Médio']:+.2f}, d_z={meninos_v2['d_z']:+.3f}). "
        f"Quanto à raça/cor, a maioria declarou-se parda (1.803; 71.3%), seguida de branca (244) e indígena (66). "
        f"Foram identificados 66 estudantes com necessidades educacionais especiais registradas, enquanto a categoria NAO_REGISTRADO (2.464 estudantes) "
        f"representa a ausência de registro administrativo e não deve ser confundida com ausência clínica comprovada de necessidade especial."
    ),
    'metricas_exatas': {
        'total_meninos_dim': 1278,
        'total_meninas_dim': 1252,
        'meninas_f2_vocab_n': int(meninas_v2['N Pares']),
        'meninas_f2_vocab_pre': float(meninas_v2['Média Pré']),
        'meninas_f2_vocab_pos': float(meninas_v2['Média Pós']),
        'meninas_f2_vocab_ganho': float(meninas_v2['Ganho Médio']),
        'meninos_f2_vocab_n': int(meninos_v2['N Pares']),
        'meninos_f2_vocab_pre': float(meninos_v2['Média Pré']),
        'meninos_f2_vocab_pos': float(meninos_v2['Média Pós']),
        'meninos_f2_vocab_ganho': float(meninos_v2['Ganho Médio']),
        'alunos_pardos': 1803,
        'alunos_necessidades_especiais_sim': 66,
        'alunos_necessidades_especiais_nao_registrado': 2464,
        'fonte': 'metricas_equidade.parquet',
    }
})

print(f"Total de {len(chunks_rag)} Chunks Semânticos sincronizados com sucesso!")

## Bloco 4: Catálogo de Avaliação e Benchmark Q&A com Ground-Truth Auditado

Definimos o banco de perguntas representativas garantindo sincronização estrita com os valores das tabelas analíticas.

In [ ]:
# Carregamento de métricas auditadas para inserção no benchmark
v2_row = df_es[(df_es['Instrumento'] == 'Vocabulário') & (df_es['Fase'] == 'Fase 2') & (df_es['Filtro'].str.contains('Completas'))].iloc[0]
t3_row = df_es[(df_es['Instrumento'] == 'TDE II') & (df_es['Fase'] == 'Fase 3') & (df_es['Filtro'].str.contains('Completas'))].iloc[0]

benchmark_qa = [
    {
        'id': 'QA-01',
        'categoria': 'Geral',
        'complexidade': 'Simples',
        'pergunta': 'Quantos estudantes únicos participaram do programa WordGen no total?',
        'resposta_ground_truth': f"Participaram exatamente {int(kpis['Estudantes Únicos (União Vocabulário e TDE)']):,} estudantes únicos na união dos instrumentos Vocabulário e TDE II (sendo {int(kpis['Estudantes Únicos em Vocabulário']):,} em Vocabulário e {int(kpis['Estudantes Únicos em TDE II']):,} em TDE II). Na dimensão socioeconômica dim_aluno constam {int(kpis['Estudantes com Cadastro Socioeconômico']):,} estudantes, com cobertura de {kpis['Cobertura Socioeconômica']}%.",
        'valor_exato': int(kpis['Estudantes Únicos (União Vocabulário e TDE)']),
        'unidade': 'Pessoas',
        'fonte': 'dashboard_kpis.parquet',
        'keywords': ['estudantes únicos', 'total de alunos', 'quantos alunos'],
    },
    {
        'id': 'QA-02',
        'categoria': 'Geral',
        'complexidade': 'Simples',
        'pergunta': 'Quantas escolas públicas municipais e turmas participaram da intervenção?',
        'resposta_ground_truth': f"Participaram {int(kpis['Escolas Públicas Atendidas'])} escolas públicas municipais identificadas e {int(kpis['Turmas em Escolas Identificadas'])} turmas escolares distintas (combinação de escola + série + turma válida). No censo geral, incluindo registros com escola não informada, há {int(kpis['Turmas Totais Registradas (incl. Não Informada)'])} turmas registradas.",
        'valor_exato': {'escolas': int(kpis['Escolas Públicas Atendidas']), 'turmas_identificadas': int(kpis['Turmas em Escolas Identificadas']), 'turmas_totais': int(kpis['Turmas Totais Registradas (incl. Não Informada)'])},
        'unidade': 'Instituições / Turmas',
        'fonte': 'dashboard_kpis.parquet',
        'keywords': ['escolas', 'quantas escolas', 'turmas atendidas'],
    },
    {
        'id': 'QA-03',
        'categoria': 'Desempenho',
        'complexidade': 'Intermediária',
        'pergunta': 'Qual foi o Effect Size (d_z de Cohen) do Vocabulário na Fase 2 para a amostra completa?',
        'resposta_ground_truth': f"Na Fase 2 de Vocabulário para a amostra completa (N={v2_row['N Pares']:,} pares com >=80% dos itens válidos), o Effect Size pareado (d_z de Cohen = Ganho Médio / s_D) foi de {v2_row['d_z (Cohen)']:+.3f} (IC 95%: [{v2_row['IC 95% Inf']:.3f}, {v2_row['IC 95% Sup']:.3f}]), com p-valor {v2_row['p-valor (t)']}, classificado qualitativamente como {v2_row['Interpretação']}.",
        'valor_exato': float(v2_row['d_z (Cohen)']),
        'unidade': 'd_z de Cohen (adimensional)',
        'fonte': 'metricas_effect_size.parquet',
        'keywords': ['effect size', 'cohen dz', 'fase 2', 'vocabulario'],
    },
    {
        'id': 'QA-04',
        'categoria': 'Desempenho',
        'complexidade': 'Intermediária',
        'pergunta': 'Qual foi a pontuação média no pré e pós-teste de Vocabulário na Fase 2?',
        'resposta_ground_truth': f"Na Fase 2 de Vocabulário (amostra completa, N={v2_row['N Pares']:,}), a pontuação média no pré-teste foi de {v2_row['Média Pré']:.2f} pontos e no pós-teste subiu para {v2_row['Média Pós']:.2f} pontos, gerando um ganho médio intra-aluno de {v2_row['Ganho Médio']:+.2f} pontos (desvio padrão das diferenças s_D = {v2_row['s_D']:.2f}).",
        'valor_exato': {'n': int(v2_row['N Pares']), 'pre': float(v2_row['Média Pré']), 'pos': float(v2_row['Média Pós']), 'ganho': float(v2_row['Ganho Médio']), 's_d': float(v2_row['s_D'])},
        'unidade': 'Pontos (Escala 0-50)',
        'fonte': 'metricas_effect_size.parquet',
        'keywords': ['média pré', 'média pós', 'fase 2', 'vocabulario'],
    },
    {
        'id': 'QA-05',
        'categoria': 'Desempenho',
        'complexidade': 'Intermediária',
        'pergunta': 'Qual foi o resultado do TDE II na Fase 3?',
        'resposta_ground_truth': f"Na Fase 3 do TDE II (amostra completa, N={t3_row['N Pares']:,} pares), a média no pré-teste foi de {t3_row['Média Pré']:.2f} pontos e no pós-teste subiu para {t3_row['Média Pós']:.2f} pontos (ganho médio de {t3_row['Ganho Médio']:+.2f} pontos; s_D = {t3_row['s_D']:.2f}). O Effect Size pareado (d_z de Cohen) foi de {t3_row['d_z (Cohen)']:+.3f} (IC 95%: [{t3_row['IC 95% Inf']:.3f}, {t3_row['IC 95% Sup']:.3f}], p < 0.001).",
        'valor_exato': {'n': int(t3_row['N Pares']), 'pre': float(t3_row['Média Pré']), 'pos': float(t3_row['Média Pós']), 'ganho': float(t3_row['Ganho Médio']), 'dz': float(t3_row['d_z (Cohen)'])},
        'unidade': 'Pontos (Escala 0-80) / d_z',
        'fonte': 'metricas_effect_size.parquet',
        'keywords': ['tde', 'fase 3', 'ganho', 'effect size'],
    },
    {
        'id': 'QA-06',
        'categoria': 'Longitudinal',
        'complexidade': 'Complexa',
        'pergunta': 'O que aconteceu com a pontuação dos estudantes entre o pós-teste da Fase 2 e o pré-teste da Fase 3 em Vocabulário?',
        'resposta_ground_truth': f"Para os {ret_v23_disp['N Pares']:,} estudantes com pontuações disponíveis em ambos os momentos (acompanhados por aluno_id), a pontuação média passou de {ret_v23_disp['Média Pós Anterior']:.2f} pontos no pós de Fase 2 para {ret_v23_disp['Média Pré Seguinte']:.2f} pontos no pré de Fase 3, uma redução de {ret_v23_disp['Variação Média (Delta)']:+.2f} pontos (t = {ret_v23_disp['t (Student)']:.2f}, p < 0.001). Na amostra com completude estrita em ambos os testes (N=520), a variação observada foi de -4.06 pontos. Essa redução observada no retorno letivo decorre potencialmente da descontinuidade escolar no recesso, mas também pode envolver fatores como calibração dos testes.",
        'valor_exato': {'n_disponivel': int(ret_v23_disp['N Pares']), 'variacao_disponivel': float(ret_v23_disp['Variação Média (Delta)']), 'n_completo': 520, 'variacao_completo': -4.06},
        'unidade': 'Pontos (Escala 0-50)',
        'fonte': 'metricas_longitudinais.parquet',
        'keywords': ['transicao', 'fase 2 para fase 3', 'retorno letivo', 'vocabulario'],
    },
    {
        'id': 'QA-07',
        'categoria': 'Longitudinal',
        'complexidade': 'Complexa',
        'pergunta': 'Por que a média global de Vocabulário na Fase 3 parece menor em relação à Fase 2?',
        'resposta_ground_truth': "A média global da Fase 3 é fortemente influenciada por um efeito de composição da amostra: 370 novos estudantes ingressaram na Fase 3 com linha de base muito baixa (média pré de 19.03 pontos), enquanto os veteranos que participaram da Fase 2 mantiveram média pré de 22.69 pontos. Assim, a média da fase reflete a entrada de novatos e não retrocesso dos veteranos.",
        'valor_exato': {'media_novatos_pre': 19.03, 'media_veteranos_pre': 22.69, 'novatos_n': 370},
        'unidade': 'Pontos',
        'fonte': 'metricas_longitudinais.parquet',
        'keywords': ['efeito composicao', 'atrito', 'novatos', 'veteranos', 'fase 3'],
    },
    {
        'id': 'QA-08',
        'categoria': 'Equidade',
        'complexidade': 'Intermediária',
        'pergunta': 'Como evoluiu a diferença de pontuação entre estudantes da zona rural e urbana na Fase 2 de Vocabulário?',
        'resposta_ground_truth': f"Na amostra de avaliações completas (>=80% itens), os estudantes da zona rural (N={eq_v2_comp['N Rural']}) tiveram ganho médio de {eq_v2_comp['Ganho Médio Rural']:+.2f} pontos (de {eq_v2_comp['Pré Rural']:.2f} para {eq_v2_comp['Pós Rural']:.2f}), superior ao ganho dos estudantes urbanos de {eq_v2_comp['Ganho Médio Urbano']:+.2f} pontos (N={eq_v2_comp['N Urbana']}, de {eq_v2_comp['Pré Urbana']:.2f} para {eq_v2_comp['Pós Urbana']:.2f}). Com isso, o gap pré-existente de {eq_v2_comp['Gap Pré (U - R)']:.2f} pontos reduziu-se para {eq_v2_comp['Gap Pós (U - R)']:.2f} pontos (Delta Gap = {eq_v2_comp['Delta Gap']:+.2f} pontos). Na amostra total de pares válidos, os ganhos foram de +1.40 (rural) e +0.70 (urbano), com Delta Gap de -0.71.",
        'valor_exato': {'amostra_completa': {'ganho_rural': float(eq_v2_comp['Ganho Médio Rural']), 'ganho_urbano': float(eq_v2_comp['Ganho Médio Urbano']), 'gap_pre': float(eq_v2_comp['Gap Pré (U - R)']), 'gap_pos': float(eq_v2_comp['Gap Pós (U - R)']), 'delta_gap': float(eq_v2_comp['Delta Gap'])}, 'amostra_total': {'ganho_rural': float(eq_v2_all['Ganho Médio Rural']), 'ganho_urbano': float(eq_v2_all['Ganho Médio Urbano']), 'delta_gap': float(eq_v2_all['Delta Gap'])}},
        'unidade': 'Pontos',
        'fonte': 'metricas_equidade.parquet',
        'keywords': ['zona rural', 'zona urbana', 'gap', 'equidade', 'disparidade'],
    },
    {
        'id': 'QA-09',
        'categoria': 'Metodologia',
        'complexidade': 'Intermediária',
        'pergunta': 'Qual é a diferença entre contagem de avaliações pareadas e contagem de estudantes únicos?',
        'resposta_ground_truth': f"No projeto WordGen foram auditadas {int(kpis['Linhas de Avaliações Pareadas Completas']):,} linhas de avaliações pareadas completas (3.320 em Vocabulário e 3.796 em TDE II). Como um mesmo estudante pode participar dos dois instrumentos e em múltiplas fases, essas 7.116 linhas correspondem a exatamente {int(kpis['Estudantes Únicos com Par Completo']):,} estudantes únicos (pessoas) com pelo menos um par completo. Não se deve confundir contagem de registros com contagem de indivíduos.",
        'valor_exato': {'linhas_pares_completos': int(kpis['Linhas de Avaliações Pareadas Completas']), 'estudantes_unicos_par_completo': int(kpis['Estudantes Únicos com Par Completo'])},
        'unidade': 'Linhas vs. Pessoas',
        'fonte': 'dashboard_kpis.parquet',
        'keywords': ['linhas pareadas', 'estudantes únicos', 'unidade de observacao'],
    },
    {
        'id': 'QA-10',
        'categoria': 'Sociodemográfico',
        'complexidade': 'Simples',
        'pergunta': 'Quantos estudantes com necessidades educacionais especiais foram identificados e o que significa NAO_REGISTRADO?',
        'resposta_ground_truth': "Foram identificados 66 estudantes com registro formal de necessidade especial (SIM). A categoria NAO_REGISTRADO (2.464 estudantes) indica a ausência de registro formal nos cadastros escolares consultados e não deve ser interpretada como confirmação clínica ou diagnóstica de ausência de necessidade especial.",
        'valor_exato': {'sim': 66, 'nao_registrado': 2464},
        'unidade': 'Pessoas',
        'fonte': 'perfil_socioeconomico_desempenho.parquet',
        'keywords': ['necessidades especiais', 'nao registrado', 'inclusao'],
    },
    {
        'id': 'QA-11',
        'categoria': 'Metodologia',
        'complexidade': 'Intermediária',
        'pergunta': 'Qual o critério técnico para considerar uma avaliação pareada como completa?',
        'resposta_ground_truth': "Uma avaliação pareada é classificada como completa quando o estudante respondeu validamente a pelo menos 80% dos itens tanto na aplicação pré-teste quanto na pós-teste (critério booleano completo_pre & completo_pos). Avaliações com preenchimento parcial (< 80%) continuam na base bruta para análises censitárias, mas são filtradas nos cálculos do padrão de referência de effect size.",
        'valor_exato': 0.80,
        'unidade': 'Proporção mínima de itens',
        'fonte': 'metricas_effect_size.parquet',
        'keywords': ['criterio completude', '80%', 'pares completos', 'regra tecnica'],
    },
    {
        'id': 'QA-12',
        'categoria': 'Desempenho',
        'complexidade': 'Intermediária',
        'pergunta': 'Como o desempenho de Vocabulário na Fase 2 variou por sexo na amostra completa?',
        'resposta_ground_truth': f"Na Fase 2 de Vocabulário para a amostra completa (>=80% itens), as meninas (N={meninas_v2['N Pares']}) obtiveram média pré de {meninas_v2['Média Pré']:.2f} e pós de {meninas_v2['Média Pós']:.2f} (ganho médio de {meninas_v2['Ganho Médio']:+.2f} pontos; d_z = {meninas_v2['d_z']:+.3f}). Os meninos (N={meninos_v2['N Pares']}) obtiveram média pré de {meninos_v2['Média Pré']:.2f} e pós de {meninos_v2['Média Pós']:.2f} (ganho médio de {meninos_v2['Ganho Médio']:+.2f} pontos; d_z = {meninos_v2['d_z']:+.3f}). Na amostra total de pares válidos, as médias pré/pós foram 27.04/28.52 (meninas) e 25.58/25.93 (meninos).",
        'valor_exato': {'amostra_completa': {'meninas_ganho': float(meninas_v2['Ganho Médio']), 'meninos_ganho': float(meninos_v2['Ganho Médio']), 'meninas_dz': float(meninas_v2['d_z']), 'meninos_dz': float(meninos_v2['d_z'])}, 'amostra_total': {'meninas_pre': 27.04, 'meninas_pos': 28.52, 'meninos_pre': 25.58, 'meninos_pos': 25.93}},
        'unidade': 'Pontos',
        'fonte': 'metricas_equidade.parquet',
        'keywords': ['sexo', 'genero', 'meninas', 'meninos', 'fase 2 vocabulario'],
    },
]

df_qa = pd.DataFrame(benchmark_qa)
print(f"Catálogo de Avaliação auditado com {len(df_qa)} perguntas e respostas sincronizadas!")
display(df_qa[['id', 'categoria', 'complexidade', 'pergunta', 'fonte']])

## Bloco 5: Exportação dos Artefatos de Conhecimento para o Agente RAG

In [ ]:
# 1. Exportar base de conhecimento JSON estruturada
json_path = DATA_RAG / 'base_conhecimento_wordgen.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump({
        'projeto': 'WordGen',
        'versao': '2.0.0_pos_auditoria',
        'data_atualizacao': '2026-10-29',
        'total_chunks': len(chunks_rag),
        'chunks': chunks_rag,
    }, f, ensure_ascii=False, indent=2)

# 2. Exportar documento consolidado em Markdown (documentos_rag.md)
md_path = DATA_RAG / 'documentos_rag.md'
with open(md_path, 'w', encoding='utf-8') as f:
    f.write("# Base de Conhecimento Analítico — Projeto WordGen\n\n")
    f.write("> Documentação estruturada auditada e sincronizada para recuperação em agentes RAG e modelos de linguagem.\n\n---\n\n")
    
    for c in chunks_rag:
        f.write(f"## [{c['chunk_id']}] {c['titulo']}\n\n")
        f.write(f"- **Categoria:** `{c['categoria']}`\n")
        f.write(f"- **Tags:** {', '.join(c['tags'])}\n\n")
        f.write(f"{c['conteudo']}\n\n")
        f.write("```json\n")
        f.write(json.dumps(c['metricas_exatas'], ensure_ascii=False, indent=2))
        f.write("\n```\n\n---\n\n")

# 3. Exportar catálogo de benchmark Ground-Truth
qa_path = DATA_RAG / 'benchmark_qa_ground_truth.json'
with open(qa_path, 'w', encoding='utf-8') as f:
    json.dump(benchmark_qa, f, ensure_ascii=False, indent=2)

print("Artefatos RAG sincronizados exportados com sucesso em data/rag/:")
print(f"  -> {json_path} ({len(chunks_rag)} chunks)")
print(f"  -> {md_path}")
print(f"  -> {qa_path} ({len(benchmark_qa)} pares de perguntas e respostas auditadas)")

## Bloco 6: Teste de Recuperação Preliminar e Limitações Metodológicas

*Aviso metodológico:* o teste de busca textual a seguir tem finalidade exclusivamente verificatória da indexação preliminar dos chunks por palavras-chave. Ele não substitui nem comprova a acurácia de geração e alinhamento de um modelo LLM completo (RAG), o qual será avaliado na etapa posterior da disciplina utilizando o benchmark de ground-truth `benchmark_qa_ground_truth.json`.

In [ ]:
def recuperar_chunk(query, chunks):
    termos = query.lower().split()
    melhor_score = -1
    melhor_chunk = None
    
    for c in chunks:
        texto = (c['titulo'] + " " + " ".join(c['tags']) + " " + c['conteudo']).lower()
        score = sum(1 for t in termos if t in texto)
        if score > melhor_score:
            melhor_score = score
            melhor_chunk = c
            
    return melhor_chunk, melhor_score

# Teste 1: Consulta sobre variação entre fases e atrito
q1 = "variacao inter-fases atrito composicao novatos veteranos"
chunk_res1, score1 = recuperar_chunk(q1, chunks_rag)
print(f"Consulta: '{q1}'")
print(f"Chunk Recuperado: [{chunk_res1['chunk_id']}] {chunk_res1['titulo']} (Score: {score1})")
print(f"Trecho: {chunk_res1['conteudo'][:180]}...")

print()
# Teste 2: Consulta sobre equidade territorial
q2 = "equidade rural urbana gap closure diferenca observada"
chunk_res2, score2 = recuperar_chunk(q2, chunks_rag)
print(f"Consulta: '{q2}'")
print(f"Chunk Recuperado: [{chunk_res2['chunk_id']}] {chunk_res2['titulo']} (Score: {score2})")
print(f"Trecho: {chunk_res2['conteudo'][:180]}...")